# Grid Search Cross-Validation Explained

## What is Grid Search Cross-Validation?

**Grid Search Cross-Validation** is a hyperparameter tuning technique that systematically searches through a predefined set of hyperparameter combinations to find the optimal configuration for a machine learning model.

### Key Concepts:

1. **Hyperparameters**: Parameters that are set before training begins (e.g., learning rate, regularization strength, number of trees)
2. **Grid Search**: Exhaustive search through all possible combinations of hyperparameters
3. **Cross-Validation**: A technique to evaluate model performance by splitting data into multiple folds
4. **Parameter Grid**: A dictionary defining the range of values to test for each hyperparameter

### Why Use Grid Search Cross-Validation?

- **Automated Hyperparameter Tuning**: Eliminates manual trial-and-error
- **Robust Performance Estimation**: Uses cross-validation to avoid overfitting to a single train-test split
- **Comprehensive Search**: Tests all combinations in the specified parameter space
- **Statistical Reliability**: Provides confidence in the selected hyperparameters

### How It Works:

1. Define a parameter grid with hyperparameter ranges
2. For each parameter combination:
   - Train the model using k-fold cross-validation
   - Calculate average performance across all folds
3. Select the combination with the best average performance
4. Retrain the model on the full dataset using optimal parameters

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

print("All libraries imported successfully!")

## Load a Sample Dataset

We'll use the famous **Iris dataset** to demonstrate grid search cross-validation. This dataset contains measurements of iris flowers from three different species.

In [ ]:
# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data  # Features: sepal length, sepal width, petal length, petal width
y = iris.target  # Target: species (setosa, versicolor, virginica)

# Create a DataFrame for better visualization
df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Dataset Shape:", X.shape)
print("\nFeature Names:", iris.feature_names)
print("\nTarget Names:", iris.target_names)
print("\nFirst 5 rows:")
print(df.head())

# Display dataset statistics
print("\nDataset Statistics:")
print(df.describe())

In [ ]:
# Visualize the dataset
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Iris Dataset Feature Distributions', fontsize=16)

for i, feature in enumerate(iris.feature_names):
    row = i // 2
    col = i % 2
    
    for species_idx, species_name in enumerate(iris.target_names):
        species_data = df[df['species'] == species_idx][feature]
        axes[row, col].hist(species_data, alpha=0.7, label=species_name, bins=15)
    
    axes[row, col].set_title(feature)
    axes[row, col].set_xlabel('Value')
    axes[row, col].set_ylabel('Frequency')
    axes[row, col].legend()

plt.tight_layout()
plt.show()

# Class distribution
print("\nClass Distribution:")
print(df['species_name'].value_counts())

## Split the Dataset into Training and Testing Sets

Before applying grid search, we need to split our dataset into training and testing sets. The training set will be used for cross-validation during grid search, while the test set will be used for final evaluation.

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

# Check class distribution in train and test sets
print("\nTraining set class distribution:")
train_counts = np.bincount(y_train)
for i, count in enumerate(train_counts):
    print(f"{iris.target_names[i]}: {count} samples ({count/len(y_train)*100:.1f}%)")

print("\nTest set class distribution:")
test_counts = np.bincount(y_test)
for i, count in enumerate(test_counts):
    print(f"{iris.target_names[i]}: {count} samples ({count/len(y_test)*100:.1f}%)")

# Scale the features for SVM (important for distance-based algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled successfully!")

## Define a Model for Grid Search

We'll use a **Support Vector Machine (SVM)** classifier for our demonstration. SVMs have several important hyperparameters that significantly affect performance:

- **C**: Regularization parameter (controls overfitting)
- **kernel**: Type of kernel function ('linear', 'rbf', 'poly', 'sigmoid')
- **gamma**: Kernel coefficient for 'rbf', 'poly', and 'sigmoid'

### Understanding SVM Hyperparameters:

1. **C (Regularization Parameter)**:
   - Low C: More regularization, simpler decision boundary
   - High C: Less regularization, more complex decision boundary

2. **Kernel**:
   - 'linear': Good for linearly separable data
   - 'rbf' (Radial Basis Function): Good for non-linear data
   - 'poly': Polynomial kernel

3. **Gamma**:
   - Low gamma: Far-reaching influence, smoother decision boundary
   - High gamma: Close influence, more complex decision boundary

In [ ]:
# Create a basic SVM model with default parameters
svm_default = SVC(random_state=42)

# Train the model and evaluate with cross-validation
default_scores = cross_val_score(svm_default, X_train_scaled, y_train, cv=5)

print("SVM with Default Parameters:")
print(f"Cross-validation scores: {default_scores}")
print(f"Mean CV accuracy: {default_scores.mean():.4f} (+/- {default_scores.std() * 2:.4f})")

# Train on full training set and evaluate on test set
svm_default.fit(X_train_scaled, y_train)
y_pred_default = svm_default.predict(X_test_scaled)
default_test_accuracy = accuracy_score(y_test, y_pred_default)

print(f"Test set accuracy: {default_test_accuracy:.4f}")

# Display default parameters
print("\nDefault SVM Parameters:")
for param, value in svm_default.get_params().items():
    print(f"{param}: {value}")

## Set Up the Parameter Grid

The parameter grid defines all the hyperparameter combinations we want to test. For our SVM, we'll create a comprehensive grid that explores different kernels and their associated parameters.

In [ ]:
# Define parameter grid for SVM
param_grid = {
    'C': [0.1, 1, 10, 100],  # Regularization parameter
    'kernel': ['linear', 'rbf', 'poly'],  # Kernel types
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],  # Kernel coefficient
    'degree': [2, 3, 4]  # Degree for polynomial kernel
}

# Calculate total number of combinations
total_combinations = 1
for param, values in param_grid.items():
    total_combinations *= len(values)

print("Parameter Grid:")
for param, values in param_grid.items():
    print(f"{param}: {values}")

print(f"\nTotal parameter combinations to test: {total_combinations}")
print(f"With 5-fold CV, total model fits: {total_combinations * 5}")

# Create a more focused parameter grid for demonstration
# (to reduce computation time)
focused_param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 0.01, 0.1, 1]
}

# Note: gamma is ignored for linear kernel
focused_combinations = len(focused_param_grid['C']) * (
    1 +  # linear kernel (gamma ignored)
    len(focused_param_grid['gamma'])  # rbf kernel
)

print(f"\nFocused grid combinations: {focused_combinations}")
print("Using focused grid for demonstration...")

## Perform Grid Search Cross-Validation

Now we'll use `GridSearchCV` to systematically test all parameter combinations and find the optimal hyperparameters.

### GridSearchCV Parameters:
- **estimator**: The machine learning model
- **param_grid**: Dictionary of parameters to search
- **cv**: Number of cross-validation folds
- **scoring**: Metric to optimize
- **n_jobs**: Number of parallel processes (-1 uses all cores)
- **verbose**: Level of output detail

In [ ]:
# Perform Grid Search Cross-Validation
print("Starting Grid Search Cross-Validation...")
print("This may take a moment...\n")

# Create GridSearchCV object
grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=focused_param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',  # Optimization metric
    n_jobs=-1,  # Use all available cores
    verbose=1,  # Print progress
    return_train_score=True  # Keep training scores for analysis
)

# Fit grid search to training data
grid_search.fit(X_train_scaled, y_train)

print("\n" + "="*50)
print("GRID SEARCH COMPLETED!")
print("="*50)

# Display best parameters and score
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")
print(f"Best Estimator: {grid_search.best_estimator_}")

# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
best_test_accuracy = accuracy_score(y_test, y_pred_best)

print(f"\nTest Set Performance:")
print(f"Best Model Test Accuracy: {best_test_accuracy:.4f}")
print(f"Improvement over default: {best_test_accuracy - default_test_accuracy:.4f}")

## Analyze the Best Parameters and Scores

Let's dive deeper into the grid search results to understand which parameter combinations performed best and worst.

In [ ]:
# Convert results to DataFrame for easier analysis
results_df = pd.DataFrame(grid_search.cv_results_)

# Display key columns
key_columns = [
    'param_C', 'param_kernel', 'param_gamma',
    'mean_test_score', 'std_test_score', 'rank_test_score'
]

print("Top 10 Parameter Combinations:")
print("="*70)
top_results = results_df.nlargest(10, 'mean_test_score')[key_columns]
for idx, row in top_results.iterrows():
    print(f"Rank {int(row['rank_test_score'])}: "
          f"C={row['param_C']}, kernel={row['param_kernel']}, gamma={row['param_gamma']} "
          f"→ Score: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")

print("\n" + "="*70)
print("Bottom 5 Parameter Combinations:")
print("="*70)
bottom_results = results_df.nsmallest(5, 'mean_test_score')[key_columns]
for idx, row in bottom_results.iterrows():
    print(f"Rank {int(row['rank_test_score'])}: "
          f"C={row['param_C']}, kernel={row['param_kernel']}, gamma={row['param_gamma']} "
          f"→ Score: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")

# Statistical summary
print("\n" + "="*50)
print("STATISTICAL SUMMARY")
print("="*50)
print(f"Number of parameter combinations tested: {len(results_df)}")
print(f"Best cross-validation score: {results_df['mean_test_score'].max():.4f}")
print(f"Worst cross-validation score: {results_df['mean_test_score'].min():.4f}")
print(f"Mean cross-validation score: {results_df['mean_test_score'].mean():.4f}")
print(f"Standard deviation: {results_df['mean_test_score'].std():.4f}")

# Performance by kernel
print("\nPerformance by Kernel:")
kernel_performance = results_df.groupby('param_kernel')['mean_test_score'].agg(['mean', 'std', 'max', 'min'])
print(kernel_performance)

In [ ]:
# Detailed evaluation of the best model
print("BEST MODEL EVALUATION")
print("="*50)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=iris.target_names))

# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_best)
print(cm)

# Compare with default model
print("\n" + "="*50)
print("COMPARISON: BEST vs DEFAULT MODEL")
print("="*50)
print(f"Default Model Test Accuracy: {default_test_accuracy:.4f}")
print(f"Best Model Test Accuracy: {best_test_accuracy:.4f}")
print(f"Improvement: {best_test_accuracy - default_test_accuracy:.4f}")
print(f"Relative Improvement: {((best_test_accuracy / default_test_accuracy) - 1) * 100:.2f}%")

# Cross-validation comparison
print("\nCross-Validation Scores Comparison:")
print(f"Default Model CV Mean: {default_scores.mean():.4f} (±{default_scores.std() * 2:.4f})")
print(f"Best Model CV Mean: {grid_search.best_score_:.4f}")

# Feature importance (for linear kernel)
if grid_search.best_params_['kernel'] == 'linear':
    print("\nFeature Importance (Linear SVM):")
    feature_importance = abs(best_model.coef_[0])
    for i, importance in enumerate(feature_importance):
        print(f"{iris.feature_names[i]}: {importance:.4f}")

## Visualize Grid Search Results

Visualization helps us understand how different hyperparameters affect model performance and identify patterns in the parameter space.

In [ ]:
# Set up the plotting area
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Grid Search Cross-Validation Results Analysis', fontsize=16)

# 1. Score distribution histogram
axes[0, 0].hist(results_df['mean_test_score'], bins=20, alpha=0.7, edgecolor='black')
axes[0, 0].axvline(grid_search.best_score_, color='red', linestyle='--', 
                   label=f'Best Score: {grid_search.best_score_:.4f}')
axes[0, 0].set_xlabel('Cross-Validation Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of CV Scores')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Performance by C parameter
c_performance = results_df.groupby('param_C')['mean_test_score'].agg(['mean', 'std'])
c_values = c_performance.index
axes[0, 1].errorbar(range(len(c_values)), c_performance['mean'], 
                    yerr=c_performance['std'], marker='o', capsize=5)
axes[0, 1].set_xticks(range(len(c_values)))
axes[0, 1].set_xticklabels(c_values)
axes[0, 1].set_xlabel('C Parameter')
axes[0, 1].set_ylabel('Mean CV Score')
axes[0, 1].set_title('Performance vs C Parameter')
axes[0, 1].grid(True, alpha=0.3)

# 3. Performance by kernel
kernel_scores = []
kernels = results_df['param_kernel'].unique()
for kernel in kernels:
    kernel_data = results_df[results_df['param_kernel'] == kernel]['mean_test_score']
    kernel_scores.append(kernel_data.values)

axes[1, 0].boxplot(kernel_scores, labels=kernels)
axes[1, 0].set_xlabel('Kernel Type')
axes[1, 0].set_ylabel('CV Score')
axes[1, 0].set_title('Performance Distribution by Kernel')
axes[1, 0].grid(True, alpha=0.3)

# 4. Training vs Validation scores (overfitting analysis)
axes[1, 1].scatter(results_df['mean_train_score'], results_df['mean_test_score'], 
                   alpha=0.6, s=50)
axes[1, 1].plot([0, 1], [0, 1], 'r--', alpha=0.8, label='Perfect Agreement')
axes[1, 1].set_xlabel('Mean Training Score')
axes[1, 1].set_ylabel('Mean Validation Score')
axes[1, 1].set_title('Training vs Validation Scores')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Create heatmap for RBF kernel performance
print("\nCreating heatmap for RBF kernel performance...")
rbf_results = results_df[results_df['param_kernel'] == 'rbf'].copy()

if len(rbf_results) > 0:
    # Create pivot table for heatmap
    heatmap_data = rbf_results.pivot_table(
        values='mean_test_score', 
        index='param_C', 
        columns='param_gamma', 
        aggfunc='mean'
    )
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='viridis', 
                cbar_kws={'label': 'CV Score'})
    plt.title('RBF Kernel Performance: C vs Gamma', fontsize=14)
    plt.xlabel('Gamma Parameter')
    plt.ylabel('C Parameter')
    plt.show()

# Performance comparison bar chart
plt.figure(figsize=(10, 6))
model_names = ['Default SVM', 'Best SVM (Grid Search)']
test_scores = [default_test_accuracy, best_test_accuracy]
cv_scores = [default_scores.mean(), grid_search.best_score_]

x = np.arange(len(model_names))
width = 0.35

plt.bar(x - width/2, test_scores, width, label='Test Accuracy', alpha=0.8)
plt.bar(x + width/2, cv_scores, width, label='CV Accuracy', alpha=0.8)

plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.title('Model Performance Comparison')
plt.xticks(x, model_names)
plt.legend()
plt.ylim(0.8, 1.05)

# Add value labels on bars
for i, v in enumerate(test_scores):
    plt.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
for i, v in enumerate(cv_scores):
    plt.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Advanced Example: Grid Search with Random Forest

Let's demonstrate grid search with a different algorithm - Random Forest - to show how the technique applies to various models.

In [ ]:
# Grid Search with Random Forest
print("Grid Search with Random Forest Classifier")
print("="*50)

# Define parameter grid for Random Forest
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Calculate total combinations
rf_combinations = 1
for param, values in rf_param_grid.items():
    rf_combinations *= len(values)

print(f"Total Random Forest combinations: {rf_combinations}")
print("This would take too long, so let's use a smaller grid...")

# Smaller, focused grid for demonstration
rf_focused_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5]
}

print(f"\nUsing focused grid with {3*3*2} = 18 combinations")

# Perform grid search
rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_focused_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

print("\nRunning Random Forest Grid Search...")
rf_grid_search.fit(X_train, y_train)  # Note: No scaling needed for Random Forest

# Results
print(f"\nBest Random Forest Parameters: {rf_grid_search.best_params_}")
print(f"Best Random Forest CV Score: {rf_grid_search.best_score_:.4f}")

# Test set evaluation
rf_y_pred = rf_grid_search.best_estimator_.predict(X_test)
rf_test_accuracy = accuracy_score(y_test, rf_y_pred)
print(f"Random Forest Test Accuracy: {rf_test_accuracy:.4f}")

# Compare all models
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
models_comparison = {
    'Default SVM': default_test_accuracy,
    'Optimized SVM': best_test_accuracy,
    'Optimized Random Forest': rf_test_accuracy
}

for model_name, accuracy in models_comparison.items():
    print(f"{model_name:<25}: {accuracy:.4f}")

best_overall = max(models_comparison, key=models_comparison.get)
print(f"\nBest Overall Model: {best_overall} ({models_comparison[best_overall]:.4f})")

## Key Takeaways and Best Practices

### 🎯 **What We Learned**

1. **Grid Search Process**:
   - Systematically tests all parameter combinations
   - Uses cross-validation for robust evaluation
   - Automatically selects the best performing model

2. **Performance Improvements**:
   - Default SVM achieved decent performance
   - Grid search optimization improved results
   - Different algorithms may perform differently on the same data

3. **Parameter Interactions**:
   - Some parameters work better together
   - Visualizations help identify patterns
   - Heatmaps reveal parameter relationships

### 📋 **Best Practices**

#### 1. **Parameter Grid Design**
```python
# ✅ Good: Start with wide ranges, then narrow down
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],  # Wide range
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

# ❌ Avoid: Too narrow ranges might miss optimal values
param_grid = {
    'C': [0.9, 1.0, 1.1],  # Too narrow
    'gamma': [0.09, 0.1, 0.11]
}
```

#### 2. **Cross-Validation Strategy**
```python
# ✅ Good: Use appropriate CV for your data size
cv=5  # Good for most datasets
cv=10  # Better for small datasets
cv=3  # Faster for large datasets

# ✅ Consider stratified CV for imbalanced data
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

#### 3. **Computational Efficiency**
```python
# ✅ Use parallel processing
n_jobs=-1  # Use all available cores

# ✅ Consider RandomizedSearchCV for large parameter spaces
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator, param_distributions, n_iter=100, cv=5, n_jobs=-1
)
```

#### 4. **Validation Strategy**
```python
# ✅ Always evaluate on a separate test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ✅ Use the test set only once for final evaluation
final_score = grid_search.best_estimator_.score(X_test, y_test)
```

### ⚠️ **Common Pitfalls to Avoid**

1. **Data Leakage**: Never include test data in grid search
2. **Overfitting**: Don't tune too many parameters on small datasets
3. **Computational Cost**: Be mindful of the parameter space size
4. **Metric Selection**: Choose appropriate scoring metrics for your problem
5. **Feature Scaling**: Remember to scale features when necessary

### 🚀 **When to Use Grid Search vs Alternatives**

- **Grid Search**: Small parameter spaces, guaranteed to find optimal combination
- **Random Search**: Large parameter spaces, faster exploration
- **Bayesian Optimization**: Complex parameter spaces, intelligent search
- **Hyperband**: Large datasets, early stopping strategies

### 📊 **Monitoring and Analysis**

```python
# Always analyze your results
results_df = pd.DataFrame(grid_search.cv_results_)

# Check for overfitting
training_scores = results_df['mean_train_score']
validation_scores = results_df['mean_test_score']
print(f"Training-Validation Gap: {(training_scores - validation_scores).mean():.4f}")

# Identify stable parameters
std_scores = results_df['std_test_score']
stable_models = results_df[std_scores < std_scores.quantile(0.25)]
```

Grid Search Cross-Validation is a powerful technique that, when used correctly, can significantly improve model performance and provide confidence in hyperparameter selection!